<a href="https://colab.research.google.com/github/nishthadighe-bit/Data--Engineering-Practicals/blob/main/Practical_9_MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import sqlite3
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, round, lit, current_timestamp

# =====================================================================
# STEP 1: INITIALIZE SPARK SESSION
# =====================================================================
print("Initializing PySpark Engine...")
!pip install pyspark --quiet

spark = SparkSession.builder \
    .appName("Practical_9_MiniProject_ETL") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version} Session Active!")

# =====================================================================
# STEP 2: DATA EXTRACTION (API + Flat File Sources)
# =====================================================================
print("\n--- [PHASE 1: EXTRACTION] ---")

# Source A: REST API Extraction (User Data)
api_url = "https://jsonplaceholder.typicode.com/users"
try:
    response = requests.get(api_url, timeout=10)
    response.raise_for_status()
    users_raw = response.json()
    df_users_pd = pd.json_normalize(users_raw)[["id", "name", "email", "address.city"]]
    df_users_pd.columns = ["user_id", "user_name", "email", "city"]
    print(f"Successfully extracted {len(df_users_pd)} records from REST API.")
except Exception as e:
    print(f"API Extraction Error: {e}")
    df_users_pd = pd.DataFrame()

# Source B: CSV Generation & Extraction (Transactions Data)
csv_filename = "transactions.csv"
transactions_mock = {
    "transaction_id": [501, 502, 503, 504, 505, 506, 507, 508, 509, 510],
    "user_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 100],  # 100 is an orphan key for validation testing
    "amount": [250.50, 1200.00, -50.00, 450.75, 3100.00, 890.20, 150.00, 2100.50, 670.00, 500.00], # -50 is invalid
    "status": ['COMPLETED', 'COMPLETED', 'FAILED', 'COMPLETED', 'COMPLETED', 'PENDING', 'COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPLETED']
}
pd.DataFrame(transactions_mock).to_csv(csv_filename, index=False)

df_txns_pd = pd.read_csv(csv_filename)
print(f"Successfully extracted {len(df_txns_pd)} records from Flat CSV File.")

# Convert Pandas DataFrames to PySpark DataFrames
spark_users = spark.createDataFrame(df_users_pd)
spark_txns = spark.createDataFrame(df_txns_pd)

# =====================================================================
# STEP 3: TRANSFORMATION & DATA QUALITY VALIDATION (PySpark)
# =====================================================================
print("\n--- [PHASE 2: TRANSFORMATION & QUALITY ASSURANCE] ---")

# Rule 1: Remove invalid transaction amounts (amount > 0)
spark_txns_clean = spark_txns.filter(col("amount") > 0)

# Rule 2: Filter for completed transactions only
spark_txns_clean = spark_txns_clean.filter(col("status") == "COMPLETED")

# Rule 3: Join Datasets (Inner Join removes orphan keys)
spark_joined = spark_txns_clean.join(spark_users, on="user_id", how="inner")

# Rule 4: Feature Engineering — Customer Tiering & Tax Calculation
spark_transformed = spark_joined \
    .withColumn("tax_amount", round(col("amount") * 0.18, 2)) \
    .withColumn("total_price", round(col("amount") + col("tax_amount"), 2)) \
    .withColumn("customer_tier",
                when(col("amount") >= 2000, "Platinum")
                .when(col("amount") >= 1000, "Gold")
                .otherwise("Standard")) \
    .withColumn("processed_timestamp", current_timestamp())

print("Transformed PySpark Data Overview:")
spark_transformed.select(
    "transaction_id", "user_name", "city", "amount", "tax_amount", "total_price", "customer_tier"
).show(5)

# =====================================================================
# STEP 4: TARGET DATA WAREHOUSE LOADING (SQLite)
# =====================================================================
print("\n--- [PHASE 3: WAREHOUSE LOADING] ---")

# Convert back to Pandas for SQLite load
final_warehouse_df = spark_transformed.toPandas()

# Format timestamp column as string for SQLite compatibility
final_warehouse_df['processed_timestamp'] = final_warehouse_df['processed_timestamp'].astype(str)

db_path = "enterprise_dw.db"
conn = sqlite3.connect(db_path)

# Load into SQLite Table
final_warehouse_df.to_sql("fact_customer_orders", conn, if_exists="replace", index=False)

print(f"Loaded {len(final_warehouse_df)} clean records into Data Warehouse table 'fact_customer_orders'.")

# Verification Query
print("\n--- [WAREHOUSE AUDIT QUERY] ---")
audit_df = pd.read_sql_query("SELECT transaction_id, user_name, city, total_price, customer_tier, processed_timestamp FROM fact_customer_orders", conn)
print(audit_df)

conn.close()
spark.stop()

print("\n🎉 Practical 9 Mini Project Completed Successfully! All 9 Practicals Finished!")

Initializing PySpark Engine...
Spark Version: 4.0.4 Session Active!

--- [PHASE 1: EXTRACTION] ---
Successfully extracted 10 records from REST API.
Successfully extracted 10 records from Flat CSV File.

--- [PHASE 2: TRANSFORMATION & QUALITY ASSURANCE] ---
Transformed PySpark Data Overview:
+--------------+----------------+-----------+------+----------+-----------+-------------+
|transaction_id|       user_name|       city|amount|tax_amount|total_price|customer_tier|
+--------------+----------------+-----------+------+----------+-----------+-------------+
|           501|   Leanne Graham|Gwenborough| 250.5|     45.09|     295.59|     Standard|
|           502|    Ervin Howell|Wisokyburgh|1200.0|     216.0|     1416.0|         Gold|
|           504|Patricia Lebsack|South Elvis|450.75|     81.13|     531.88|     Standard|
|           505|Chelsey Dietrich| Roscoeview|3100.0|     558.0|     3658.0|     Platinum|
|           507| Kurtis Weissnat|  Howemouth| 150.0|      27.0|      177.0|   